In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
from wordcloud import WordCloud, STOPWORDS, ImageColorGenerator
import nltk
import re
from nltk.corpus import stopwords
import string

data = pd.read_csv("jobs.csv")
print(data.head())

   Unnamed: 0                    Job Salary Job Experience Required  \
0           0   Not Disclosed by Recruiter               5 - 10 yrs   
1           1   Not Disclosed by Recruiter                2 - 5 yrs   
2           2   Not Disclosed by Recruiter                0 - 1 yrs   
3           3       2,00,000 - 4,00,000 PA.               0 - 5 yrs   
4           4   Not Disclosed by Recruiter                2 - 5 yrs   

                                          Key Skills  \
0                      Media Planning| Digital Media   
1   pre sales| closing| software knowledge| clien...   
2   Computer science| Fabrication| Quality check|...   
3                                  Technical Support   
4   manual testing| test engineering| test cases|...   

                                Role Category  \
0                                 Advertising   
1                                Retail Sales   
2                                         R&D   
3  Admin/Maintenance/Security/Datawareho

In [2]:
data = data.drop("Unnamed: 0",axis=1)

In [3]:
data.isnull().sum()

Job Salary                 0
Job Experience Required    0
Key Skills                 0
Role Category              0
Functional Area            0
Industry                   0
Job Title                  0
dtype: int64

In [ ]:
text = " ".join(i for i in data["Key Skills"])
stopwords = set(STOPWORDS)
wordcloud = WordCloud(stopwords=stopwords, 
                      background_color="white").generate(text)
plt.figure( figsize=(15,10))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis("off")
plt.show()

In [ ]:
text = " ".join(i for i in data["Functional Area"])
stopwords = set(STOPWORDS)
wordcloud = WordCloud(stopwords=stopwords, 
                      background_color="white").generate(text)
plt.figure( figsize=(15,10))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis("off")
plt.show()

In [ ]:
text = " ".join(i for i in data["Job Title"])
stopwords = set(STOPWORDS)
wordcloud = WordCloud(stopwords=stopwords, 
                      background_color="white").generate(text)
plt.figure( figsize=(15,10))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis("off")
plt.show()

In [4]:
# from sklearn.feature_extraction import text
# feature = data["Key Skills"].tolist()
# tfidf = text.TfidfVectorizer(input=feature, stop_words="english")
# tfidf_matrix = tfidf.fit_transform(feature)
# similarity = cosine_similarity(tfidf_matrix)


from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Assuming `data["Key Skills"]` contains your text data
feature = data["Key Skills"].tolist()

# Initialize the TfidfVectorizer
tfidf = TfidfVectorizer(stop_words="english")

# Fit and transform your data
tfidf_matrix = tfidf.fit_transform(feature)

# Compute cosine similarity
similarity = cosine_similarity(tfidf_matrix)


In [5]:
indices = pd.Series(data.index, index=data['Job Title']).drop_duplicates()

In [6]:
def jobs_recommendation(Title, similarity = similarity):
    index = indices[Title]
    similarity_scores = list(enumerate(similarity[index]))
    similarity_scores = sorted(similarity_scores, key=lambda x: x[::], reverse=True)
    similarity_scores = similarity_scores[0:5]
    newsindices = [i[0] for i in similarity_scores]
    return data[['Job Title', 'Job Experience Required', 
                 'Key Skills']].iloc[newsindices]

In [7]:

print(jobs_recommendation("Software Developer"))

                                       Job Title Job Experience Required  \
6249          Sales/Business Development Manager               4 - 5 yrs   
6248                          Software Developer               2 - 5 yrs   
6247  Associate/Senior Associate -(NonTechnical)              5 - 10 yrs   
6246                          Software Developer               1 - 6 yrs   
6245  Associate/Senior Associate -(NonTechnical)               1 - 4 yrs   

                                             Key Skills  
6249   Networking| Printing| Aerospace| Raw material...  
6248   PHP| MVC| Laravel| AWS| SDLC| Wordpress| LAMP...  
6247   Data analysis| Investment banking| Financial ...  
6246   Coding| Wordpress| Commerce| HTML| Troublesho...  
6245   client servicing| client support| background ...  


In [ ]:
import csv
from googletrans import Translator, LANGUAGES

# Создаем объект переводчика
translator = Translator()

def translate_text(text, dest_language="ru"):
    try:
        # Пытаемся перевести текст
        translation = translator.translate(text, dest=dest_language)
        return translation.text
    except Exception as e:
        print(f"Ошибка при переводе: {e}")
        return text

def translate_csv(input_file, output_file, language="ru"):
    with open(input_file, mode='r', encoding='utf-8') as infile, \
         open(output_file, mode='w', encoding='utf-8', newline='') as outfile:

        reader = csv.reader(infile)
        writer = csv.writer(outfile)

        for row in reader:
            translated_row = [translate_text(cell, dest_language=language) for cell in row]
            writer.writerow(translated_row)
            print(f"Переведенная строка: {translated_row}")  # Для отслеживания прогресса

if __name__ == "__main__":
    input_csv = "jobs.csv"  # Укажите путь к вашему файлу CSV
    output_csv = "new.csv"  # Путь для сохранения переведенного файла
    translate_csv(input_csv, output_csv)


In [1]:
#!/usr/bin/env python
# coding: utf-8
import os
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import joblib

class JobRecommender:
    def __init__(self):
        # Load data and train the model upon initialization
        self.data = self.load_data("jobs.csv")
        self.tfidf_matrix, self.similarity = self.train_model()

    def load_data(self, file_path):
        data = pd.read_csv(file_path)
        data = data.drop("Unnamed: 0", axis=1)
        return data

    def train_model(self):
        # Assuming `data["Key Skills"]` contains your text data
        feature = self.data["Key Skills"].tolist()
        tfidf = TfidfVectorizer(stop_words="english")
        tfidf_matrix = tfidf.fit_transform(feature)
        similarity = cosine_similarity(tfidf_matrix)
        return tfidf_matrix, similarity

    def save_model(self, model_path="job_recommender_model.pkl"):
        # Save the model using joblib
        joblib.dump((self.tfidf_matrix, self.similarity), model_path)
        print("Model saved successfully!")

    def load_model(self, model_path="job_recommender_model.pkl"):
        # Load the model using joblib
        self.tfidf_matrix, self.similarity = joblib.load(model_path)
        print("Model loaded successfully!")

    def jobs_recommendation(self, title):
        # Get the index of the job title
        index = self.data[self.data['Job Title'] == title].index[0]
        # Get similarity scores for the given job title
        similarity_scores = list(enumerate(self.similarity[index]))
        # Sort the scores in descending order
        similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)
        # Extract the indices of top 5 similar jobs (excluding the same job)
        similar_jobs_indices = [i[0] for i in similarity_scores[1:6]]
        # Return the top 5 similar jobs as a JSON object
        return self.data.iloc[similar_jobs_indices].to_json(orient="records")

# Пример использования для обучения и сохранения модели:
if __name__ == "__main__":
    # Create an instance of the JobRecommender class
    recommender = JobRecommender()
    
    # Train the model
    recommender.train_model()
    
    # Save the model
    recommender.save_model()


Model saved successfully!
